Notebook: 11_final_results.ipynb

Purpose: Convert calibrated confidence probabilities into operational decisions and derive thresholds from validation evidence.

Inputs:
- confidence_predictions.parquet
- confidence_features.parquet

Outputs:
- decision_results.parquet

# 11 — Decision Model

Optimize thresholds empirically from validation data and assign ACCEPT, REVIEW, or REJECT labels.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd().parent / 'src'))

root_dir = Path.cwd().parent
artifacts_dir = root_dir / 'artifacts'
preds = pd.read_parquet(artifacts_dir / 'confidence_predictions.parquet')
conf = pd.read_parquet(artifacts_dir / 'confidence_features.parquet')

if 'confidence_label' not in conf.columns:
    conf['confidence_label'] = ((conf['mean_lead_agreement'] >= 0.7) & (conf['mean_beat_agreement'] >= 0.7) & (conf['mean_bsqi'] >= 0.6) & (conf['mean_wsqi'] >= 0.6)).astype(int)

merged = preds.merge(conf[['record_id','confidence_label']], on='record_id', how='left')
merged['confidence_label'] = merged['confidence_label'].fillna(0).astype(int)
probs = merged['confidence_probability'].astype(float).values
labels = merged['confidence_label'].values

thresholds = []
for low in np.linspace(0.0, 0.4, 5):
    for high in np.linspace(0.6, 1.0, 5):
        if high <= low:
            continue
        decision = np.where(probs >= high, 'ACCEPT', np.where(probs >= low, 'REVIEW', 'REJECT'))
        accept_mask = decision == 'ACCEPT'
        review_mask = decision == 'REVIEW'
        reject_mask = decision == 'REJECT'
        false_accept_rate = float(np.sum((labels == 0) & accept_mask) / max(1, np.sum(accept_mask)))
        error_capture_rate = float(np.sum((labels == 0) & (review_mask | reject_mask)) / max(1, np.sum(labels == 0)))
        thresholds.append({
            'low': float(low),
            'high': float(high),
            'false_accept_rate': false_accept_rate,
            'error_capture_rate': error_capture_rate,
        })

threshold_df = pd.DataFrame(thresholds)
selected = threshold_df.sort_values(['false_accept_rate', 'error_capture_rate']).iloc[0]
low = selected['low']
high = selected['high']
merged['decision_class'] = np.where(merged['confidence_probability'] >= high, 'ACCEPT', np.where(merged['confidence_probability'] >= low, 'REVIEW', 'REJECT'))
output = merged[['record_id','confidence_probability','decision_class']].copy()
output['pipeline_version'] = 'v1.0.0'
output.to_parquet(artifacts_dir / 'decision_results.parquet', index=False)
print('Wrote decision_results.parquet')
